# Objectif 4 — Segmentation d'instance avec Mask R-CNN

Ce notebook entraîne (optionnellement) et déploie le modèle de segmentation d'instance utilisé dans l'onglet **4** de l'application Flask.

- Le notebook peut être lancé depuis la racine du dépôt, le dossier `notebooks/` ou Google Colab.
- Pour exécuter un entraînement, placez `RoadAnomaly.zip` dans `data/` (ou à la racine de Colab) et mettez `ENTRAINER = True` dans la cellule suivante. Un GPU est vivement conseillé.
- Par défaut, le notebook recharge le modèle déjà entraîné quand il est présent, fait un test d'inférence, puis prépare `application/modeles/objectif_4/` pour le site.
- Dépendances locales : `pip install -r application/requirements.txt`.


In [ ]:
import json
import random
import shutil
import time
import zipfile
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


def trouver_racine_projet():
    candidats = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    for dossier in candidats:
        if (dossier / "application").is_dir() and (dossier / "notebooks").is_dir():
            return dossier
    raise FileNotFoundError(
        "Racine du dépôt introuvable. Ouvrez ce notebook depuis le dépôt du projet."
    )


RACINE = trouver_racine_projet()
DOSSIER_MODELE_APP = RACINE / "application" / "modeles" / "objectif_4"
CANDIDATS_MODELE = (
    DOSSIER_MODELE_APP / "model.pt",
    RACINE / "modeles" / "objectif_4" / "model.pt",
)
CHEMIN_MODELE = next((p for p in CANDIDATS_MODELE if p.is_file()), None)
CHEMIN_CONFIG = (
    CHEMIN_MODELE.with_name("config.json") if CHEMIN_MODELE else None
)

# Passer à True pour réentraîner Mask R-CNN ; False réutilise le checkpoint existant.
ENTRAINER = False
EPOCHS = 3
TAILLE_MIN = 400
TAILLE_MAX = 700
SEUIL_CONFIANCE = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Projet : {RACINE}")
print(f"PyTorch : {torch.__version__} | appareil : {DEVICE}")
print(f"Checkpoint existant : {CHEMIN_MODELE if CHEMIN_MODELE else 'aucun'}")

In [ ]:
ANOMALIE_SEMANTIQUE = 2
AIRE_MINIMALE = 50
EXTENSIONS_IMAGE = {".webp", ".jpg", ".jpeg", ".png"}


def trouver_dossier_frames():
    dossiers = [RACINE / "data" / "RoadAnomaly", Path("/content/RoadAnomaly")]
    for base in dossiers:
        candidats = [base / "RoadAnomaly" / "frames", base / "frames"]
        candidats.extend(base.rglob("frames") if base.exists() else [])
        for frames in candidats:
            if frames.is_dir() and any(frames.glob("*.labels")):
                return frames

    archives = [
        RACINE / "data" / "RoadAnomaly.zip",
        RACINE / "RoadAnomaly.zip",
        Path("/content/RoadAnomaly.zip"),
    ]
    archive = next((p for p in archives if p.is_file()), None)
    if archive is None:
        raise FileNotFoundError(
            "Dataset introuvable. Déposez RoadAnomaly.zip dans data/ puis relancez."
        )

    destination = (RACINE / "data" / "RoadAnomaly").resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as fichier_zip:
        for membre in fichier_zip.infolist():
            cible = (destination / membre.filename).resolve()
            try:
                cible.relative_to(destination)
            except ValueError as exc:
                raise ValueError("L'archive contient un chemin de fichier invalide.") from exc
        fichier_zip.extractall(destination)
    return trouver_dossier_frames()


def trouver_image(label_dir):
    base = label_dir.with_suffix("")
    for image in base.parent.glob(base.name + ".*"):
        if image.suffix.lower() in EXTENSIONS_IMAGE:
            return image
    raise FileNotFoundError(f"Image originale introuvable pour {label_dir.name}.")


def _creer_instance(masque):
    ys, xs = np.where(masque)
    return {
        "mask": masque.astype(np.uint8),
        "box": [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1],
    }


def charger_instances(label_dir):
    chemin_instances = label_dir / "labels_instance.png"
    chemin_semantique = label_dir / "labels_semantic.png"
    instances = []

    if chemin_instances.is_file():
        masque = cv2.imread(str(chemin_instances), cv2.IMREAD_UNCHANGED)
        if masque is None:
            raise ValueError(f"Masque illisible : {chemin_instances}")
        if masque.ndim == 3:
            masque = masque[..., 0]
        for identifiant in np.unique(masque):
            if identifiant == 0:
                continue
            objet = masque == identifiant
            if int(objet.sum()) >= AIRE_MINIMALE:
                instances.append(_creer_instance(objet))
    elif chemin_semantique.is_file():
        masque = cv2.imread(str(chemin_semantique), cv2.IMREAD_UNCHANGED)
        if masque is None:
            raise ValueError(f"Masque illisible : {chemin_semantique}")
        if masque.ndim == 3:
            masque = masque[..., 0]
        _, composantes, statistiques, _ = cv2.connectedComponentsWithStats(
            (masque == ANOMALIE_SEMANTIQUE).astype(np.uint8), connectivity=8
        )
        for identifiant in range(1, composantes):
            if statistiques[identifiant, cv2.CC_STAT_AREA] >= AIRE_MINIMALE:
                instances.append(_creer_instance(composantes == identifiant))
    return instances


print("Préparation des annotations prête. Le dataset sera lu uniquement si l'entraînement est activé.")

In [ ]:
class RoadAnomalyInstances(Dataset):
    def __init__(self, dossiers, entrainement=False):
        self.dossiers = list(dossiers)
        self.entrainement = entrainement

    def __len__(self):
        return len(self.dossiers)

    def __getitem__(self, index):
        label_dir = self.dossiers[index]
        image = cv2.imread(str(trouver_image(label_dir)), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Image illisible pour {label_dir.name}.")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]
        objets = charger_instances(label_dir)
        masques = []
        boites = []
        for objet in objets:
            masque = objet["mask"]
            if masque.shape != (height, width):
                masque = cv2.resize(
                    masque, (width, height), interpolation=cv2.INTER_NEAREST
                )
            masques.append(masque)
            boites.append(objet["box"])

        if masques:
            masques = np.stack(masques)
            boites = np.asarray(boites, dtype=np.float32)
        else:
            masques = np.zeros((0, height, width), dtype=np.uint8)
            boites = np.zeros((0, 4), dtype=np.float32)
        image = torch.from_numpy(image.copy()).permute(2, 0, 1).float().div(255)
        masques = torch.from_numpy(masques)
        boites = torch.from_numpy(boites)
        labels = torch.ones((len(boites),), dtype=torch.int64)

        if self.entrainement and random.random() < 0.5:
            image = image.flip(-1)
            masques = masques.flip(-1)
            boites = torch.stack(
                [width - boites[:, 2], boites[:, 1], width - boites[:, 0], boites[:, 3]],
                dim=1,
            )

        cible = {
            "boxes": boites,
            "labels": labels,
            "masks": masques,
            "image_id": torch.tensor(index, dtype=torch.int64),
            "area": masques.flatten(1).sum(1).float(),
            "iscrowd": torch.zeros((len(boites),), dtype=torch.int64),
        }
        return image, cible


def collate(batch):
    return tuple(zip(*batch))


def creer_modele(weights=None):
    modele = maskrcnn_resnet50_fpn(
        weights=weights,
        weights_backbone=None,
        min_size=TAILLE_MIN,
        max_size=TAILLE_MAX,
    )
    in_features_box = modele.roi_heads.box_predictor.cls_score.in_features
    modele.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes=2)
    in_features_mask = modele.roi_heads.mask_predictor.conv5_mask.in_channels
    modele.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, dim_reduced=256, num_classes=2
    )
    return modele


def evaluer(modele, chargeur):
    modele.eval()
    vrais_positifs = faux_positifs = faux_negatifs = 0
    ious = []
    with torch.inference_mode():
        for images, cibles in chargeur:
            predictions = modele([image.to(DEVICE) for image in images])
            for prediction, cible in zip(predictions, cibles):
                selection = prediction["scores"].cpu() >= SEUIL_CONFIANCE
                masques_predits = prediction["masks"][selection, 0].cpu() >= 0.5
                masques_reels = cible["masks"].bool()
                utilises = set()
                for masque_reel in masques_reels:
                    meilleur_iou, meilleur_index = 0.0, -1
                    for index, masque_predit in enumerate(masques_predits):
                        if index in utilises:
                            continue
                        intersection = (masque_reel & masque_predit).sum().item()
                        union = (masque_reel | masque_predit).sum().item()
                        iou = intersection / union if union else 0.0
                        if iou > meilleur_iou:
                            meilleur_iou, meilleur_index = iou, index
                    if meilleur_iou >= 0.5:
                        vrais_positifs += 1
                        utilises.add(meilleur_index)
                        ious.append(meilleur_iou)
                    else:
                        faux_negatifs += 1
                        ious.append(0.0)
                faux_positifs += len(masques_predits) - len(utilises)
    precision = vrais_positifs / max(1, vrais_positifs + faux_positifs)
    rappel = vrais_positifs / max(1, vrais_positifs + faux_negatifs)
    f1 = 2 * precision * rappel / max(1e-12, precision + rappel)
    return {"mIoU": float(np.mean(ious)) if ious else 0.0, "F1": f1}


In [ ]:
if ENTRAINER:
    frames_dir = trouver_dossier_frames()
    label_dirs = sorted(
        path for path in frames_dir.glob("*.labels") if path.is_dir()
    )
    dossiers_valides = [path for path in label_dirs if charger_instances(path)]
    if len(dossiers_valides) < 2:
        raise ValueError("Il faut au moins deux images annotées avec une anomalie pour entraîner.")

    random.seed(42)
    random.shuffle(dossiers_valides)
    nombre_validation = max(1, round(0.2 * len(dossiers_valides)))
    validation = dossiers_valides[:nombre_validation]
    entrainement = dossiers_valides[nombre_validation:]
    chargeur_entrainement = DataLoader(
        RoadAnomalyInstances(entrainement, entrainement=True),
        batch_size=1,
        shuffle=True,
        collate_fn=collate,
        num_workers=0,
    )
    chargeur_validation = DataLoader(
        RoadAnomalyInstances(validation),
        batch_size=1,
        shuffle=False,
        collate_fn=collate,
        num_workers=0,
    )

    modele = creer_modele(MaskRCNN_ResNet50_FPN_Weights.DEFAULT).to(DEVICE)
    for parametre in modele.backbone.parameters():
        parametre.requires_grad = False
    optimiseur = torch.optim.SGD(
        (p for p in modele.parameters() if p.requires_grad),
        lr=0.005,
        momentum=0.9,
        weight_decay=5e-4,
    )
    meilleur_miou = -1.0
    configuration = {"seuil_confiance": SEUIL_CONFIANCE, "modele": "Mask R-CNN ResNet50-FPN"}
    DOSSIER_MODELE_APP.mkdir(parents=True, exist_ok=True)

    for epoch in range(EPOCHS):
        modele.train()
        perte_totale = 0.0
        debut = time.time()
        for images, cibles in chargeur_entrainement:
            images = [image.to(DEVICE) for image in images]
            cibles = [
                {nom: valeur.to(DEVICE) for nom, valeur in cible.items()}
                for cible in cibles
            ]
            pertes = sum(modele(images, cibles).values())
            optimiseur.zero_grad()
            pertes.backward()
            optimiseur.step()
            perte_totale += pertes.item()

        metriques = evaluer(modele, chargeur_validation)
        print(
            f"Époque {epoch + 1}/{EPOCHS} | "
            f"perte {perte_totale / len(chargeur_entrainement):.3f} | "
            f"mIoU {metriques['mIoU']:.3f} | F1 {metriques['F1']:.3f} | "
            f"{time.time() - debut:.0f} s"
        )
        if metriques["mIoU"] > meilleur_miou:
            meilleur_miou = metriques["mIoU"]
            configuration.update(metriques)
            torch.save(modele.cpu().state_dict(), DOSSIER_MODELE_APP / "model.pt")
            modele.to(DEVICE)
else:
    if CHEMIN_MODELE is None:
        raise FileNotFoundError(
            "Aucun modèle existant. Déposez le dataset et passez ENTRAINER = True."
        )
    configuration = (
        json.loads(CHEMIN_CONFIG.read_text(encoding="utf-8"))
        if CHEMIN_CONFIG and CHEMIN_CONFIG.is_file()
        else {"seuil_confiance": SEUIL_CONFIANCE, "modele": "Mask R-CNN ResNet50-FPN"}
    )
    modele = creer_modele().to(DEVICE)
    etat = torch.load(CHEMIN_MODELE, map_location=DEVICE, weights_only=True)
    modele.load_state_dict(etat)
    modele.eval()
    print("Checkpoint chargé :", CHEMIN_MODELE)


In [ ]:
# Vérification d'inférence, puis export dans le dossier lu par application/app.py.
modele.eval()
with torch.inference_mode():
    resultat_test = modele([torch.zeros((3, 288, 512), device=DEVICE)])[0]
assert {"boxes", "scores", "masks"}.issubset(resultat_test)
assert resultat_test["masks"].shape[-2:] == (288, 512)
print("Test d'inférence réussi.")

DOSSIER_MODELE_APP.mkdir(parents=True, exist_ok=True)
CHEMIN_EXPORT = DOSSIER_MODELE_APP / "model.pt"
torch.save(modele.cpu().state_dict(), CHEMIN_EXPORT)
configuration.setdefault("seuil_confiance", SEUIL_CONFIANCE)
configuration.setdefault("modele", "Mask R-CNN ResNet50-FPN")
configuration.setdefault("classes", ["fond", "anomalie"])
(DOSSIER_MODELE_APP / "config.json").write_text(
    json.dumps(configuration, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Modèle prêt pour le site :", CHEMIN_EXPORT)
print("Configuration prête pour le site :", DOSSIER_MODELE_APP / "config.json")